## assignment 1

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from soft_marginSVM import LinearSVM

In [3]:
from pathlib import Path
from PIL import Image


def load_split(split_dir, image_size=(128, 128)):
    """
    Load one split (train/val/test) from folder structure:
    split_dir/
      NORMAL/
      PNEUMONIA/

    Returns
    -------
    X : np.ndarray, shape (n_samples, image_size[0] * image_size[1])
        Flattened grayscale images in [0, 1].
    y : np.ndarray, shape (n_samples,)
        Labels with NORMAL=-1, PNEUMONIA=1.
    """
    split_path = Path(split_dir)

    class_to_label = {
        "NORMAL": -1,
        "PNEUMONIA": 1,
    }

    X, y = [], []

    for class_name, label in class_to_label.items():
        class_dir = split_path / class_name
        for img_path in class_dir.glob("*.*"):
            try:
                img = Image.open(img_path).convert("L")  # grayscale
                img = img.resize(image_size)
                arr = np.asarray(img, dtype=np.float32) / 255.0
                X.append(arr.flatten())
                y.append(label)
            except Exception:
                # Skip corrupted/unreadable files
                continue

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int32)
    return X, y


base_dir = Path("chest_xray")
X_train, y_train = load_split(base_dir / "train")
X_val, y_val = load_split(base_dir / "val")
X_test, y_test = load_split(base_dir / "test")

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

X_train: (5216, 16384) y_train: (5216,)
X_val: (16, 16384) y_val: (16,)
X_test: (624, 16384) y_test: (624,)


In [4]:
from sklearn.preprocessing import StandardScaler

# Fit scaler on train only, then apply to val/test to avoid data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Scaled shapes:", X_train_scaled.shape, X_val_scaled.shape, X_test_scaled.shape)

Scaled shapes: (5216, 16384) (16, 16384) (624, 16384)


In [ ]:
import time
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# Train custom LinearSVM on the full standardized training set
X_train_custom = X_train_scaled
y_train_custom = y_train

model = LinearSVM(learning_rate=0.01, n_iterations=1000, C=1.0)
start = time.perf_counter()
model.fit(X_train_custom, y_train_custom)
train_time = time.perf_counter() - start

# Evaluate on full validation and test sets (standardized)
y_val_custom = model.predict(X_val_scaled)
y_test_custom = model.predict(X_test_scaled)

val_metrics = {
    "precision": precision_score(y_val, y_val_custom, pos_label=1),
    "recall": recall_score(y_val, y_val_custom, pos_label=1),
    "f1": f1_score(y_val, y_val_custom, pos_label=1),
    "accuracy": accuracy_score(y_val, y_val_custom),
}
test_metrics = {
    "precision": precision_score(y_test, y_test_custom, pos_label=1),
    "recall": recall_score(y_test, y_test_custom, pos_label=1),
    "f1": f1_score(y_test, y_test_custom, pos_label=1),
    "accuracy": accuracy_score(y_test, y_test_custom),
}

print(f"Custom SVM train samples: {X_train_custom.shape[0]} / {X_train_scaled.shape[0]}")
print(f"Training time: {train_time:.2f}s")

print("Validation metrics")
print(f"Precision: {val_metrics['precision']:.4f}")
print(f"Recall   : {val_metrics['recall']:.4f}")
print(f"F1-score : {val_metrics['f1']:.4f}")
print(f"Accuracy : {val_metrics['accuracy']:.4f}")

print("\nTest metrics")
print(f"Precision: {test_metrics['precision']:.4f}")
print(f"Recall   : {test_metrics['recall']:.4f}")
print(f"F1-score : {test_metrics['f1']:.4f}")
print(f"Accuracy : {test_metrics['accuracy']:.4f}")

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# Train sklearn SVM on the same full standardized training set
X_train_sk = X_train_scaled
y_train_sk = y_train

sk_model = LinearSVC(C=1.0, max_iter=200, random_state=42)
sk_model.fit(X_train_sk, y_train_sk)

y_val_pred = sk_model.predict(X_val_scaled)
y_test_pred = sk_model.predict(X_test_scaled)

print(f"[sklearn] Train samples: {X_train_sk.shape[0]} / {X_train_scaled.shape[0]}")
print("[sklearn] Validation metrics")
print(f"Precision: {precision_score(y_val, y_val_pred, pos_label=1):.4f}")
print(f"Recall   : {recall_score(y_val, y_val_pred, pos_label=1):.4f}")
print(f"F1-score : {f1_score(y_val, y_val_pred, pos_label=1):.4f}")
print(f"Accuracy : {accuracy_score(y_val, y_val_pred):.4f}")

print("\n[sklearn] Test metrics")
print(f"Precision: {precision_score(y_test, y_test_pred, pos_label=1):.4f}")
print(f"Recall   : {recall_score(y_test, y_test_pred, pos_label=1):.4f}")
print(f"F1-score : {f1_score(y_test, y_test_pred, pos_label=1):.4f}")
print(f"Accuracy : {accuracy_score(y_test, y_test_pred):.4f}")

[sklearn] Train samples: 160 / 5216
[sklearn] Validation metrics
Precision: 0.8333
Recall   : 0.6250
F1-score : 0.7143
Accuracy : 0.7500

[sklearn] Test metrics
Precision: 0.8374
Recall   : 0.7923
F1-score : 0.8142
Accuracy : 0.7740


In [ ]:
# Compare custom SVM vs sklearn SVM on val/test
sk_val_metrics = {
    "precision": precision_score(y_val, y_val_pred, pos_label=1),
    "recall": recall_score(y_val, y_val_pred, pos_label=1),
    "f1": f1_score(y_val, y_val_pred, pos_label=1),
    "accuracy": accuracy_score(y_val, y_val_pred),
}

sk_test_metrics = {
    "precision": precision_score(y_test, y_test_pred, pos_label=1),
    "recall": recall_score(y_test, y_test_pred, pos_label=1),
    "f1": f1_score(y_test, y_test_pred, pos_label=1),
    "accuracy": accuracy_score(y_test, y_test_pred),
}

metrics = ["precision", "recall", "f1", "accuracy"]

print("=== Validation Comparison (custom vs sklearn) ===")
for m in metrics:
    diff = sk_val_metrics[m] - val_metrics[m]
    print(f"{m:9s}: custom={val_metrics[m]:.4f} | sklearn={sk_val_metrics[m]:.4f} | delta={diff:+.4f}")

print("\n=== Test Comparison (custom vs sklearn) ===")
for m in metrics:
    diff = sk_test_metrics[m] - test_metrics[m]
    print(f"{m:9s}: custom={test_metrics[m]:.4f} | sklearn={sk_test_metrics[m]:.4f} | delta={diff:+.4f}")

=== Validation Comparison (custom vs sklearn) ===
precision: custom=0.8333 | sklearn=0.8333 | delta=+0.0000
recall   : custom=0.6250 | sklearn=0.6250 | delta=+0.0000
f1       : custom=0.7143 | sklearn=0.7143 | delta=+0.0000
accuracy : custom=0.7500 | sklearn=0.7500 | delta=+0.0000

=== Test Comparison (custom vs sklearn) ===
precision: custom=0.8311 | sklearn=0.8374 | delta=+0.0063
recall   : custom=0.7949 | sklearn=0.7923 | delta=-0.0026
f1       : custom=0.8126 | sklearn=0.8142 | delta=+0.0016
accuracy : custom=0.7708 | sklearn=0.7740 | delta=+0.0032
